#  Computer Vision

## What is a Convolutional Neural Network?

The dense (fully connected) networks we built in the neural networks notebook treat an image as a flat vector of pixels. A 28×28 image becomes 784 inputs, and every neuron in the first layer connects to every pixel. Two problems:

1. **No spatial awareness.** The network treats pixel (0,0) and pixel (27,27) as completely independent inputs with no sense that they might be neighbors of the same edge.
2. **Not translation invariant.** A cat in the top-left corner and a cat in the bottom-right corner produce completely different activation patterns, so the network has to learn the cat concept separately for every position.

Convolutional Neural Networks (CNNs) solve both problems with two key layer types.

## Convolutional Layers

A convolutional layer slides a small **filter** (e.g. 3×3 pixels) across the input image. At each position, it computes a dot product between the filter weights and the patch of pixels underneath, producing a single number. Sliding across the whole image produces a **feature map** — one value per position.

Key properties:
- **Local patterns:** Each filter only looks at a small neighbourhood at a time. Early filters learn low-level features (edges, colours); deeper filters combine those into higher-level patterns (eyes, wheels).
- **Shared weights:** The same filter weights are applied everywhere in the image. The network learns one edge detector and applies it at every location — much fewer parameters than a dense layer.
- **Translation invariance:** Because the same filter scans the whole image, it can detect a feature (e.g. a diagonal edge) regardless of where it appears.

A convolutional layer typically learns many filters in parallel (32, 64, 128…), each capturing a different pattern. The output has shape **(height × width × n_filters)**.

### Visualising Convolution

The cell below shows how a single 3×3 filter moves across a small 5×5 input to produce a 3×3 feature map. Use the slider to step through each filter position.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets

# Simple 5x5 input (a rough diagonal)
input_img = np.array([
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1],
], dtype=float)

# Diagonal-edge detector filter
filt = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
], dtype=float)

# Compute 3x3 feature map
feature_map = np.zeros((3, 3))
for r in range(3):
    for c in range(3):
        feature_map[r, c] = np.sum(input_img[r:r+3, c:c+3] * filt)

# List positions in row-major order
positions = [(r, c) for r in range(3) for c in range(3)]

def show_conv_step(step):
    r, c = positions[step]
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))

    # Input with filter position highlighted
    axes[0].imshow(input_img, cmap='Blues', vmin=0, vmax=1)
    rect = patches.Rectangle((c-0.5, r-0.5), 3, 3, linewidth=2,
                               edgecolor='red', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].set_title('Input image\n(red = filter position)')
    axes[0].axis('off')

    # Filter
    axes[1].imshow(filt, cmap='Oranges', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            axes[1].text(j, i, f'{filt[i,j]:.0f}', ha='center', va='center', fontsize=12)
    axes[1].set_title('Filter (3×3)\ndot product with patch')
    axes[1].axis('off')

    # Feature map — reveal up to current step
    fm_display = np.full((3, 3), np.nan)
    for k in range(step + 1):
        pr, pc = positions[k]
        fm_display[pr, pc] = feature_map[pr, pc]
    axes[2].imshow(fm_display, cmap='Greens', vmin=0, vmax=3)
    for i in range(3):
        for j in range(3):
            if not np.isnan(fm_display[i, j]):
                axes[2].text(j, i, f'{fm_display[i,j]:.0f}', ha='center', va='center', fontsize=12)
    axes[2].set_title(f'Feature map\n(step {step+1}/9)')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

widgets.interact(show_conv_step,
                 step=widgets.IntSlider(value=0, min=0, max=8, step=1,
                                        description='Step',
                                        style={'description_width': 'initial'}));

## Max Pooling

After a convolutional layer, we typically apply a **max pooling** layer. It slides a window (usually 2×2) across the feature map and takes the maximum value in each window, then moves by a stride of 2. This halves the spatial dimensions.

Why? Two reasons:
1. **Reduces computation** — the feature maps get progressively smaller as the network deepens.
2. **Adds translation invariance** — if a feature shifted by one pixel, max pooling will likely still fire at the same location.

A typical CNN architecture alternates: Conv → MaxPool → Conv → MaxPool → Flatten → Dense → Output.

## What Each Layer Learns

CNN layers form a **spatial hierarchy of patterns**:

- **Early layers:** Simple local features — edges, corners, colour gradients
- **Middle layers:** Combinations of those — textures, simple shapes (circles, lines)
- **Deep layers:** High-level concepts — faces, wheels, fur, text

This is analogous to how our visual cortex is believed to work: V1 responds to edges, V2 to curves and corners, and higher areas to object parts and whole objects. In the cats-and-dogs example below, we will actually visualise what each layer activates on a specific image.

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import optimizers
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import matplotlib.pyplot as plt

#### MNIST image set

We'll revisit a familiar exmaple, the MNIST image set. We have a 28x28 black and white pixel representation of 70,000 digits.

As a first step, we'll construct a convolutional neural network:

In [ ]:
# Initialize sequential layer model
model = models.Sequential()

# Add several convolutional and pooling layers
model.add(layers.Conv2D(32, (3, 3), activation = 'relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation = 'relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation = 'relu'))

In [ ]:
model.summary()

What are these layers doing? We start with images that are 28x28x1. We then add a 3x3 convolutional layer, with depth 32. Note that the size of our image representation shrinks to 26x26. The max pooling layer looks at 2x2 blocks (non-overlapping) and uses the max value on the block. That's why the dimensions are cut in half. Our next 3x3 convolutional layer reduces each dimension by 2, and the pooling layer cuts them in half again. The final convolution reduces our image to a 3x3 representation, with a depth of 64 layers.

Next, we'll add a classifier:

In [ ]:
# Flatten the final output and classify using two dense layers
model.add(layers.Flatten())
model.add(layers.Dense(64, activation = 'relu'))
model.add(layers.Dense(10, activation = 'softmax'))

Dense layers connect every input node to every node in the layer. Our last convolution ended with 3x3x64 nodes, or 576 nodes after flattening. Each of those is an input to our next layer of 64 nodes, and each of those 64 is an input into our final layer of 10 nodes. The last layer has 'softmax' as an activation function, which will return the most likely class.

In [ ]:
model.summary()

We can train the model on MNIST images:

In [ ]:
from keras.datasets import mnist
from keras.utils import to_categorical

In [ ]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

In [ ]:
# Reshape the data to pixel diagrams
train_images = train_images.reshape((60000, 28, 28, 1))
train_images = train_images.astype('float32') / 255

test_images = test_images.reshape((10000, 28, 28, 1))
test_images = test_images.astype('float32') / 255

# Map the image labels to categorical variables
train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

In [ ]:
# Train the model
model.compile(optimizer = 'rmsprop', loss = 'categorical_crossentropy', metrics = ['accuracy'])
model.fit(train_images, train_labels, epochs = 5, batch_size = 64)

Finally, we'll evaluate the model on the test data:

In [ ]:
test_acc = model.evaluate(test_images, test_labels)[1]
test_acc

Our test accuracy is around 99% which is much better than the 93.4% we got out of our simple multilayer perceptron.

#### Cats and Dogs

We want to train a model that tells if a photo contains a cat or a dog. The data comes from this Kaggle competition: https://www.kaggle.com/c/dogs-vs-cats/data which has 25,000 images. For cats and dogs, we will use the first 1000 as a training set, the next 500 for validation, and the next 500 for test. The following code downloads the subsets of the original training data. Since this is a balanced binary classification problem, accuracy will be an appropriate measure of success.

##### Organizaing the data

In [ ]:
import zipfile
import os

# Name and Create Directories

# Base Directory
base_dir = '/tmp/cats_and_dogs'
os.mkdir(base_dir)

# Directories for our training, validation and test splits
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')
os.mkdir(train_dir)
os.mkdir(validation_dir)
os.mkdir(test_dir)

# Directory with our training cat pictures
train_cats_dir = os.path.join(train_dir, 'cats')
os.mkdir(train_cats_dir)

# Directory with our training dog pictures
train_dogs_dir = os.path.join(train_dir, 'dogs')
os.mkdir(train_dogs_dir)

# Directory with our validation cat pictures
validation_cats_dir = os.path.join(validation_dir, 'cats')
os.mkdir(validation_cats_dir)

# Directory with our validation dog pictures
validation_dogs_dir = os.path.join(validation_dir, 'dogs')
os.mkdir(validation_dogs_dir)

# Directory with our test cat pictures
test_cats_dir = os.path.join(test_dir, 'cats')
os.mkdir(test_cats_dir)

# Directory with our test dog pictures
test_dogs_dir = os.path.join(test_dir, 'dogs')
os.mkdir(test_dogs_dir)

In [ ]:
# Download the data
sets = ['train_cats', 'train_dogs', 'validation_cats', 'validation_dogs', 'test_cats', 'test_dogs']

!wget 'https://github.com/dsahota-applied-data-analysis/data/blob/main/cats_and_dogs/train_cats.zip?raw=true' -O '/tmp/cats_and_dogs/train_cats.zip'
!wget 'https://github.com/dsahota-applied-data-analysis/data/blob/main/cats_and_dogs/train_dogs.zip?raw=true' -O '/tmp/cats_and_dogs/train_dogs.zip'
!wget 'https://github.com/dsahota-applied-data-analysis/data/blob/main/cats_and_dogs/validation_cats.zip?raw=true' -O '/tmp/cats_and_dogs/validation_cats.zip'
!wget 'https://github.com/dsahota-applied-data-analysis/data/blob/main/cats_and_dogs/validation_dogs.zip?raw=true' -O '/tmp/cats_and_dogs/validation_dogs.zip'
!wget 'https://github.com/dsahota-applied-data-analysis/data/blob/main/cats_and_dogs/test_cats.zip?raw=true' -O '/tmp/cats_and_dogs/test_cats.zip'
!wget 'https://github.com/dsahota-applied-data-analysis/data/blob/main/cats_and_dogs/test_dogs.zip?raw=true' -O '/tmp/cats_and_dogs/test_dogs.zip'

In [ ]:
# Extract the Data
for s in sets:
    zip_ref = zipfile.ZipFile('/tmp/cats_and_dogs/' + s + '.zip', 'r')
    zip_ref.extractall(globals()[s + '_dir'])
    zip_ref.close()

In [ ]:
# As a sanity check, we can veryify how many pictures are in each set
print('total training cat images:', len(os.listdir(train_cats_dir)))
print('total training dog images:', len(os.listdir(train_dogs_dir)))
print('total validation cat images:', len(os.listdir(validation_cats_dir)))
print('total validation dog images:', len(os.listdir(validation_dogs_dir)))
print('total test cat images:', len(os.listdir(test_cats_dir)))
print('total test dog images:', len(os.listdir(test_dogs_dir)))

We'll build a CNN similar to the MNIST example, but because we have a more complex problem and larger images, we'll use a larger network (one additional set of layers). This increases the capacity of the network, but also further reduces the size of the feature maps before we reach the Flatten layer. It is common for the depth of feature maps to increase progressively (here from 32 to 128), while the size of the feature maps decreases (here from 150x150 to 7x7).

In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))

model.summary()

In [ ]:
model.compile(loss = 'binary_crossentropy', optimizer = optimizers.RMSprop(learning_rate=1e-4), metrics = ['acc'])

##### A detour to discuss Generators

A generator is a function that acts as an iterator, by leveraging the yield operator. You can use it with the for ... in operator

In [ ]:
def generator():
    i = 0
    while True:
        i += 1
        yield i

for item in generator():
    print(item)
    if item > 4:
        break

##### Data preprocessing

The images currently sit on the drive as jpeg files, so we need to do a bit of preprocessing: read the picture files, decode jpeg to rbg pixels, convert to floats, and rescale to [0,1]. Keras has tools to help us do this, using generators.

In [ ]:
# All images will be rescaled by 1./255
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
        # This is the target directory
        train_dir,
        # All images will be resized to 150x150
        target_size=(150, 150),
        batch_size=20,
        # Since we use binary_crossentropy loss, we need binary labels
        class_mode='binary')

validation_generator = test_datagen.flow_from_directory(
        validation_dir,
        target_size=(150, 150),
        batch_size=20,
        class_mode='binary')

Each of these generators produces batches of 20 150x150 RGB images (shape (20, 150, 150, 3)) and binary labels (shape (20,)). 20 is the number of samples in each batch (the batch size). The generator yields these batches indefinitely: it loops endlessly over the images in the target folder. For this reason, we need to break the iteration loop at some point.

In [ ]:
for data_batch, labels_batch in train_generator:
    print('data batch shape:', data_batch.shape)
    print('labels batch shape:', labels_batch.shape)
    break

We'll fit our model using the generator. The fit method can actually take a generator, yielding batches of inputs and targets indefinitely. Because the data is being generated endlessly, the generator needs to knowhow many samples to drawbefore declaring an epoch over. We can use steps_per_epoch argument: after drawing steps_per_epoch batches from the generator, the fitting process will go to the next epoch. In our case, batches are 20-sample large, so it will take 100 batches until we see our target of 2000 samples.

We can also pass a validation_data argument. This argument is allowed to be a data generator, but it could be a tuple of Numpy arrays as well. If you pass a generator as validation_data, you should also specify the validation_steps argument, specifying how many batches to draw from the validation generator for evaluation. We will use 50 steps, to cover the 1000 validation samples.

In [ ]:
# Caution, this cell can take up to 1 hour to run.
history = model.fit(
      train_generator,
      steps_per_epoch=100,
      epochs=30,
      validation_data=validation_generator,
      validation_steps=50)

In [ ]:
# Of course, we should save the model after training
model.save(os.path.join(base_dir, 'cats_and_dogs_small_model_1.keras'))

You may be able to tell from our training and validation plots (both accuracy and validation) that we have an overfitting issue. This makes sense, because we only used 2000 images for training.

In [ ]:
acc = history.history['acc']
val_acc = history.history['val_acc']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.plot(epochs, acc, 'bo', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.legend()

plt.figure()

plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()

plt.show()

##### Data augmentation

To mitigate this, we would like more training data. Suppose this was all the data we had. We can generate additional data by taking our existing pictures and rotating them, shifting them up, down, left, or right, shearing them, zooming them, or flipping them. We'll generate some augmented images and view one set.

In [ ]:
datagen = ImageDataGenerator(
      rotation_range=40,
      width_shift_range=0.2,
      height_shift_range=0.2,
      shear_range=0.2,
      zoom_range=0.2,
      horizontal_flip=True,
      fill_mode='nearest')

In [ ]:
fnames = [os.path.join(train_cats_dir, fname) for fname in os.listdir(train_cats_dir)]

# We pick one image to "augment"
img_path = fnames[3]

# Read the image and resize it
img = image.load_img(img_path, target_size=(150, 150))

# Convert it to a Numpy array with shape (150, 150, 3)
x = image.img_to_array(img)

# Reshape it to (1, 150, 150, 3)
x = x.reshape((1,) + x.shape)

# The .flow() command below generates batches of randomly transformed images.
# It will loop indefinitely, so we need to `break` the loop at some point!
i = 0
for batch in datagen.flow(x, batch_size=1):
    plt.figure(i)
    imgplot = plt.imshow(image.array_to_img(batch[0]))
    i += 1
    if i % 4 == 0:
        break

plt.show()

With this data augmentation configuration, our network will never see the same input twice. However, the inputs that it sees are still heavily intercorrelated, since they come from a small number of original images. This is because we cannot produce new information, we can only remix existing information. This might not be quite enough to completely get rid of overfitting. To further fight overfitting, we will also add a Dropout layer to our model, right before the densely-connected classifier:

In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dropout(0.5))
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer=optimizers.RMSprop(learning_rate=1e-4),
              metrics=['acc'])

In [ ]:
model.summary()

Next, we'll train our new model with augmentation and dropout.

In [ ]:
# Please note that this cell can take up to 4 hours to run.

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest')

# Note that the validation data should not be augmented!
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
        # This is the target directory
        train_dir,
        # All images will be resized to 150x150
        target_size=(150, 150),
        batch_size=20,
        # Since we use binary_crossentropy loss, we need binary labels
        class_mode='binary')

validation_generator = test_datagen.flow_from_directory(
        validation_dir,
        target_size=(150, 150),
        batch_size=20,
        class_mode='binary')

history = model.fit(
      train_generator,
      steps_per_epoch=100,
      epochs=100,
      validation_data=validation_generator,
      validation_steps=50)

In [ ]:
# That took a while; we better save this one too!
model.save(os.path.join(base_dir, 'cats_and_dogs_small_model_2.keras'))

Let's take another look at these plots.

In [ ]:
acc = history.history['acc']
val_acc = history.history['val_acc']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.plot(epochs, acc, 'bo', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.legend()

plt.figure()

plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()

plt.show()

Because of the data augmentation and dropout, we seem to have less of an issue of overfitting.

Next, let's see what the CNN actually learns. We'll start with an image of a cat that was not part of the images the network was trained on.

In [ ]:
img_path = '/tmp/cats_and_dogs/test/cats/cat.1700.jpg'

# We preprocess the image into a 4D tensor
import numpy as np

img = image.load_img(img_path, target_size=(150, 150))
img_tensor = image.img_to_array(img)
img_tensor = np.expand_dims(img_tensor, axis=0)
# Remember that the model was trained on inputs that were preprocessed in the following way:
img_tensor /= 255.

# Its shape is (1, 150, 150, 3)
print(img_tensor.shape)

Here is our cat

In [ ]:
plt.imshow(img_tensor[0])
plt.show()

Next we'll create a model which takes an image as input and returns the layer activations from the original model.

In [ ]:
# Extracts the outputs of the top 8 layers:
layer_outputs = [layer.output for layer in model.layers[:8]]
# Creates a model that will return these outputs, given the model input:
activation_model = models.Model(inputs=model.inputs, outputs=layer_outputs)

In [ ]:
# This will return a list of 5 Numpy arrays: one array per layer activation
activations = activation_model.predict(img_tensor)

In [ ]:
# For instance, this is the activation of the first convolution layer for our cat image input:
first_layer_activation = activations[0]
print(first_layer_activation.shape)

This first layer is 148x148 with 32 channels; we can visualize the 3rd and 30th channels (yours may vary since the specific filters learned are not deterministic.)

In [ ]:
# Let's visualize the 3rd channel:
plt.matshow(first_layer_activation[0, :, :, 3], cmap='viridis')
plt.show()

In [ ]:
# This is the 30th channel:
plt.matshow(first_layer_activation[0, :, :, 30], cmap='viridis')
plt.show()

In fact, we can visualize all of the activations in the network.

In [ ]:
# These are the names of the layers, so can have them as part of our plot
layer_names = []
for layer in model.layers[:8]:
    layer_names.append(layer.name)

images_per_row = 16

# Now let's display our feature maps
for layer_name, layer_activation in zip(layer_names, activations):
    # This is the number of features in the feature map
    n_features = layer_activation.shape[-1]

    # The feature map has shape (1, size, size, n_features)
    size = layer_activation.shape[1]

    # We will tile the activation channels in this matrix
    n_cols = n_features // images_per_row
    display_grid = np.zeros((size * n_cols, images_per_row * size))

    # We'll tile each filter into this big horizontal grid
    for col in range(n_cols):
        for row in range(images_per_row):
            channel_image = layer_activation[0,
                                             :, :,
                                             col * images_per_row + row]
            # Post-process the feature to make it visually palatable
            channel_image -= channel_image.mean()
            channel_image /= channel_image.std()
            channel_image *= 64
            channel_image += 128
            channel_image = np.clip(channel_image, 0, 255).astype('uint8')
            display_grid[col * size : (col + 1) * size,
                         row * size : (row + 1) * size] = channel_image

    # Display the grid
    scale = 1. / size
    plt.figure(figsize=(scale * display_grid.shape[1],
                        scale * display_grid.shape[0]))
    plt.title(layer_name)
    plt.grid(False)
    plt.imshow(display_grid, aspect='auto', cmap='viridis')

plt.show()

As discussed in class, the features extracted get increasingly abstract with the depth of the layer.